In [1]:
from pyspark.sql import SparkSession
from pyspark.mllib.tree import RandomForest
from pyspark.mllib.regression import LabeledPoint
from pyspark.mllib.evaluation import MulticlassMetrics
from pyspark.sql.functions import when, col, isnan, isnull
from pyspark.mllib.linalg import Vectors
import numpy as np

In [2]:
spark = SparkSession.builder \
    .appName("RandomForestMLlib") \
    .getOrCreate()

sc = spark.sparkContext
data = spark.read.csv('data.csv', header=True, inferSchema=True)

In [3]:
# Show first few rows
data.show(5)

# Check for null values in diagnosis column
print(f"\nNull values in diagnosis column: {data.filter(data.diagnosis.isNull()).count()}")

# Check unique values in diagnosis column
print("\nUnique values in diagnosis column:")
data.select("diagnosis").distinct().show()

# Check target distribution
data.groupBy("diagnosis").count().show()

+--------+---------+-----------+------------+--------------+---------+---------------+----------------+--------------+-------------------+-------------+----------------------+---------+----------+------------+-------+-------------+--------------+------------+-----------------+-----------+--------------------+------------+-------------+---------------+----------+----------------+-----------------+---------------+--------------------+--------------+-----------------------+
|      id|diagnosis|Radius_mean|Texture_mean|perimeter_mean|area_mean|smoothness_mean|compactness_mean|concavity_mean|concave points_mean|symmetry_mean|fractal_dimension_mean|radius_se|texture_se|perimeter_se|area_se|smoothness_se|compactness_se|concavity_se|concave points_se|symmetry_se|fractal_dimension_se|radius_worst|texture_worst|perimeter_worst|area_worst|smoothness_worst|compactness_worst|concavity_worst|concave points_worst|symmetry_worst|fractal_dimension_worst|
+--------+---------+-----------+------------+---

In [4]:
data_labeled = data.withColumn("label", 
                              when(data.diagnosis == "M", 1.0)
                              .when(data.diagnosis == "B", 0.0)
                              .otherwise(None))

# Remove rows where diagnosis couldn't be converted (invalid values)
data_labeled = data_labeled.filter(data_labeled.label.isNotNull())

print(f"\nRows after filtering valid diagnosis values: {data_labeled.count()}")


Rows after filtering valid diagnosis values: 569


In [6]:
column_types = dict(data_labeled.dtypes)
print(f"\nColumn types: {column_types}")

# Drop the original diagnosis column and select only numeric columns
numeric_types = ['int', 'bigint', 'float', 'double']
feature_cols = [col_name for col_name, col_type in column_types.items() 
                if col_name not in ['diagnosis', 'label', 'id'] and col_type in numeric_types]

print(f"\nUsing {len(feature_cols)} numeric features for training")
print("Feature columns:", feature_cols[:10], "..." if len(feature_cols) > 10 else "")


Column types: {'id': 'int', 'diagnosis': 'string', 'Radius_mean': 'double', 'Texture_mean': 'double', 'perimeter_mean': 'double', 'area_mean': 'double', 'smoothness_mean': 'double', 'compactness_mean': 'double', 'concavity_mean': 'double', 'concave points_mean': 'double', 'symmetry_mean': 'double', 'fractal_dimension_mean': 'double', 'radius_se': 'double', 'texture_se': 'double', 'perimeter_se': 'double', 'area_se': 'double', 'smoothness_se': 'double', 'compactness_se': 'double', 'concavity_se': 'double', 'concave points_se': 'double', 'symmetry_se': 'double', 'fractal_dimension_se': 'double', 'radius_worst': 'double', 'texture_worst': 'double', 'perimeter_worst': 'double', 'area_worst': 'double', 'smoothness_worst': 'double', 'compactness_worst': 'double', 'concavity_worst': 'double', 'concave points_worst': 'double', 'symmetry_worst': 'double', 'fractal_dimension_worst': 'double', 'label': 'double'}

Using 30 numeric features for training
Feature columns: ['Radius_mean', 'Texture_me

In [7]:
model_data = data_labeled.select(feature_cols + ['label'])

# Check for null values in features and label
print(f"\nChecking for null values in selected columns...")
for col_name in feature_cols + ['label']:
    null_count = model_data.filter(col(col_name).isNull() | isnan(col(col_name))).count()
    if null_count > 0:
        print(f"Column '{col_name}' has {null_count} null/NaN values")


Checking for null values in selected columns...


In [8]:
model_data_clean = model_data.dropna()
rows_before = model_data.count()
rows_after = model_data_clean.count()
print(f"\nRows before cleaning: {rows_before}")
print(f"Rows after cleaning: {rows_after}")
print(f"Rows removed: {rows_before - rows_after}")


Rows before cleaning: 569
Rows after cleaning: 569
Rows removed: 0


In [13]:
print(f"\nExamining sample data before RDD conversion:")
sample_rows = model_data.take(3)
for i, row in enumerate(sample_rows):
    print(f"Row {i}: {row}")


Examining sample data before RDD conversion:
Row 0: Row(Radius_mean=17.99, Texture_mean=10.38, perimeter_mean=122.8, area_mean=1001.0, smoothness_mean=0.1184, compactness_mean=0.2776, concavity_mean=0.3001, concave points_mean=0.1471, symmetry_mean=0.2419, fractal_dimension_mean=0.07871, radius_se=1.095, texture_se=0.9053, perimeter_se=8.589, area_se=153.4, smoothness_se=0.006399, compactness_se=0.04904, concavity_se=0.05373, concave points_se=0.01587, symmetry_se=0.03003, fractal_dimension_se=0.006193, radius_worst=25.38, texture_worst=17.33, perimeter_worst=184.6, area_worst=2019.0, smoothness_worst=0.1622, compactness_worst=0.6656, concavity_worst=0.7119, concave points_worst=0.2654, symmetry_worst=0.4601, fractal_dimension_worst=0.1189, label=1.0)
Row 1: Row(Radius_mean=20.57, Texture_mean=21.77, perimeter_mean=132.9, area_mean=1326.0, smoothness_mean=0.08474, compactness_mean=0.07864, concavity_mean=0.0869, concave points_mean=0.07017, symmetry_mean=0.1812, fractal_dimension_mean

In [14]:
print(f"\nFeature columns to use: {feature_cols}")
print(f"Number of feature columns: {len(feature_cols)}")


Feature columns to use: ['Radius_mean', 'Texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst']
Number of feature columns: 30


In [15]:
total_rows = model_data.count()
print(f"Total rows in cleaned data: {total_rows}")

if total_rows == 0:
    raise ValueError("No data remaining after cleaning. Check your data file and column names.")

Total rows in cleaned data: 569


In [16]:
def create_labeled_point(row):
    try:
        # Convert row to dict for easier access
        row_dict = row.asDict()
        
        # Extract features
        features = []
        for col_name in feature_cols:
            if col_name not in row_dict:
                print(f"Warning: Column '{col_name}' not found in row")
                return None
            
            val = row_dict[col_name]
            if val is None or val == 'null':
                print(f"Warning: None/null value found in column '{col_name}'")
                return None
            
            # Convert to float, handling various data types
            try:
                features.append(float(val))
            except (ValueError, TypeError) as e:
                print(f"Warning: Cannot convert '{val}' to float in column '{col_name}': {e}")
                return None
        
        # Extract label
        if 'label' not in row_dict:
            print("Warning: 'label' column not found in row")
            return None
            
        label_val = row_dict['label']
        if label_val is None:
            print("Warning: None value found in label column")
            return None
            
        try:
            label = float(label_val)
        except (ValueError, TypeError) as e:
            print(f"Warning: Cannot convert label '{label_val}' to float: {e}")
            return None
        
        # Create LabeledPoint
        return LabeledPoint(label, Vectors.dense(features))
        
    except Exception as e:
        print(f"Error creating LabeledPoint: {e}")
        import traceback
        traceback.print_exc()
        return None

In [39]:
print(f"\nTesting LabeledPoint creation on first row...")
first_row = model_data.take(1)[0]
test_point = create_labeled_point(first_row)
if test_point is not None:
    print(f"Success! Created LabeledPoint with label={test_point.label}, features shape={len(test_point.features)}")
else:
    print("Failed to create LabeledPoint from first row")
    raise ValueError("Cannot create LabeledPoint from data. Check your data format.")

test_point


Testing LabeledPoint creation on first row...
Success! Created LabeledPoint with label=1.0, features shape=30


LabeledPoint(1.0, [17.99,10.38,122.8,1001.0,0.1184,0.2776,0.3001,0.1471,0.2419,0.07871,1.095,0.9053,8.589,153.4,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.1189])

In [ ]:
raw_rdd = model_data.rdd
print(f"Raw RDD created successfully")
mapped_rdd = raw_rdd.map(create_labeled_point)
print(f"Mapped RDD created successfully")

Raw RDD created successfully
Mapped RDD created successfully


In [35]:
print(f"\nCounting valid samples...")
try:
    total_samples = rdd_data.count()
    print(f"Total valid samples: {total_samples}")
except Exception as e:
    print(f"Error during count: {e}")
    print("Trying to collect first few elements to debug...")
    try:
        sample_elements = rdd_data.take(5)
        print(f"Successfully collected {len(sample_elements)} sample elements")
        for i, elem in enumerate(sample_elements):
            print(f"Element {i}: label={elem.label}, features_length={len(elem.features)}")
        # Now try count again
        total_samples = rdd_data.count()
        print(f"Total valid samples: {total_samples}")
    except Exception as e2:
        print(f"Error even with take(): {e2}")
        raise



Counting valid samples...
Error during count: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 131.0 failed 1 times, most recent failure: Lost task 0.0 in stage 131.0 (TID 93) (DESKTOP-E6MA3J1 executor driver): org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "C:\Users\Jason\anaconda3\Lib\site-packages\pyspark\python\lib\pyspark.zip\pyspark\worker.py", line 1100, in main
pyspark.errors.exceptions.base.PySparkRuntimeError: [PYTHON_VERSION_MISMATCH] Python in worker has different version (3, 10) than that in driver 3.12, PySpark cannot run with different minor versions.
Please check environment variables PYSPARK_PYTHON and PYSPARK_DRIVER_PYTHON are correctly set.

	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:572)
	at org.apache.spark.api.python.PythonRunner$$anon$3.re

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.runJob.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 132.0 failed 1 times, most recent failure: Lost task 0.0 in stage 132.0 (TID 94) (DESKTOP-E6MA3J1 executor driver): org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "C:\Users\Jason\anaconda3\Lib\site-packages\pyspark\python\lib\pyspark.zip\pyspark\worker.py", line 1100, in main
pyspark.errors.exceptions.base.PySparkRuntimeError: [PYTHON_VERSION_MISMATCH] Python in worker has different version (3, 10) than that in driver 3.12, PySpark cannot run with different minor versions.
Please check environment variables PYSPARK_PYTHON and PYSPARK_DRIVER_PYTHON are correctly set.

	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:572)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:784)
	at org.apa